In [1]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc
mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
H    1.746241653903   -0.373945564047    0.758561000000
'''
mol.basis = 'cc-pvdz'
# mol.precision = 1e-10
# mol.spin = 3
mol.build()
mf = scf.UHF(mol).density_fit()
mf = mf.newton()
mf.kernel()

frozen = 2
mymp = mp.MP2(mf, frozen=frozen)
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf, frozen=frozen)
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Sun Jun 14 13:27:06 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 20
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [2]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}') 

MP2 Corr = -0.40609900
CCSD Corr = -0.42461939
CCSD(T) Corr = -0.43105817


In [17]:
from afqmc.lno_afqmc import lno_afqmc, tools
iao_coeff, iao_frag_list, atm_center = tools.iao_localization(mf)

In [18]:
print(iao_coeff[0].shape)
print(iao_coeff[1].shape)

(48, 14)
(48, 14)


In [23]:
import jax
jax.config.update("jax_enable_x64", True)
from afqmc.lno_afqmc import prep, lno_afqmc
from pyscf import lib
from pyscf.lno import lnoccsd, ulnoccsd
from jax import random
from collections.abc import Iterable
from afqmc.lno_afqmc import prep, tools
from afqmc.lno_afqmc import mod_lnoccsd

from functools import partial
import time, gc, pickle
import os

In [ ]:
def run_afqmc(mf,
              lo_coeff = None, 
              frag_lolist = None,
              nfrozen = 0,
              thresh = 1e-6,
              qmc_options = {}, 
              chol_cut = 1e-5, 
              target_sto_error = 1e-3, 
              run_frag_list = None, 
              atom_group = None,
              plot_las = False,
              ):

    spin_type = prep.kind(lo_coeff)

    if frag_lolist is None:
        if spin_type == "unrestricted":
            raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
        print("Fragment list not found. Asign every LO to a fragment.")
        frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

    if spin_type == "unrestricted":
        mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
        mf = mlno._scf
    else:
        mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mlno.lno_thresh = [thresh*10, thresh]
    lno_thresh = mlno.lno_thresh
    lno_type = ['1h','1h']
    eris = mlno.ao2mo()

    nfrag_tot = len(frag_lolist)
    if run_frag_list is None:
        run_frag_list = range(nfrag_tot)
    
    frag_lolist = [frag_lolist[i] for i in run_frag_list]
    nfrag_run = len(frag_lolist)

    lno_pct_occ = [None, None]
    lno_norb = [[None,None]] * nfrag_tot

    seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                           shape=(nfrag_tot,), 
                           minval=0, 
                           maxval=100*nfrag_tot
                           )
    
    qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
    trial_base = qmc_options.get("trial", "")

    las_center = [None]*nfrag_run
    las_size = [None]*nfrag_run
    # las_size = np.zeros(nfrag, dtype='int32')
    lno_emp2 = np.zeros(nfrag_run, dtype='float64')
    lno_ecc  = np.zeros(nfrag_run, dtype='float64')
    lno_eqmc = np.zeros(nfrag_run, dtype='float64')
    lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
    ccsd_time = np.zeros(nfrag_run, dtype='float64')
    qmc_time = np.zeros(nfrag_run, dtype='float64')

    mol = mf.mol

    # Loop over fragment
    for ifrag, loidx in enumerate(frag_lolist):
        print("\n")
        width = 80
        msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/({nfrag_run},{nfrag_tot}) "
        print(msg.center(width, '='))
        if atom_group is not None:
            loc_ctr = f"{atom_group[run_frag_list[ifrag]]}"
            print(f"Center Atom {loc_ctr}")
        else:
            loc_ctr = None
        
        print(f"PySCF NumPy Threads = {lib.num_threads()}")

        if spin_type == "unrestricted":
            orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
            lno_param = [
                [
                    {
                        'thresh': (
                            lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                            else lno_thresh[i]
                        ),
                        'pct_occ': (
                            lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                            else lno_pct_occ[i]
                        ),
                        'norb': (
                            lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                            else lno_norb[ifrag][i]
                        ),
                    } for i in [0, 1] # occ, vir
                ] for s in range(2) # alpha, beta
            ]
        else:
            orbloc = lo_coeff[:,loidx]
            lno_param = [{
                'thresh': lno_thresh[i],
                'pct_occ': lno_pct_occ[i],
                'norb': lno_norb[ifrag][i]
                } for i in [0,1]]
            
        print(lno_param)

        # M = <orbloc|canactocc> (M^dagger M)u = eu
        # u|canactocc> => orbtial in/out the space spanned by |orbloc>
        # uocc_loc = <lno_actocc|orbloc>
        lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

        mo_occ = mlno.mo_occ

        if spin_type == "unrestricted":
            if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
                lno_elec_type = 'alpha'
            elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
                lno_elec_type = 'beta'
            else:
                lno_elec_type = 'mixed'
            print(f'LNO-Frgament Spin Type = {lno_elec_type}')

            if loc_ctr is None:
                ao_max_a = prep.ao_comp(mf, orbloc[0])
                ao_max_b = prep.ao_comp(mf, orbloc[1])
                loc_ctr = ao_max_a + ao_max_b
                print(f"LNO Center {loc_ctr}")

            lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
            occidxa = mo_occ[0] > 1e-10
            occidxb = mo_occ[1] > 1e-10
            moidxa, moidxb = maskact
            nactocc_a = int(np.sum(moidxa & occidxa))
            nactvir_a = int(np.sum(moidxa & ~occidxa))
            nactocc_b = int(np.sum(moidxb & occidxb))
            nactvir_b = int(np.sum(moidxb & ~occidxb))
            nactocc = [nactocc_a, nactocc_b]
            nactvir = [nactvir_a, nactvir_b]
            lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
            lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
            lno_active = [lno_active_a, lno_active_b]
            lno_tot = [len(lno_active_a), len(lno_active_b)]
            # print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
            print(f'LAS occupied orbitals:  {nactocc}')
            print(f'LAS virtual orbitals:   {nactvir}')
            print(f'LAS total size:         {lno_tot}')
            mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
        else:
            print(f'LNO-Frgament Spin Type = restricted')
            if loc_ctr is None:
                loc_ctr = prep.ao_comp(mf, orbloc)
                print(f"LNO Center {loc_ctr}")

            lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
            lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
            nactocc, nactvir = prep.las_size(mf, lno_frozen)
            lno_tot = len(lno_active)
            print(f'LAS occupied orbitals:  {nactocc}')
            print(f'LAS virtual orbitals:   {nactvir}')
            print(f'LAS total size:         {lno_tot}')
            mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
        
        if plot_las:
            tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx = run_frag_list[ifrag]+1)

        mcc._s1e = mlno._s1e
        mcc._h1e = mlno._h1e
        mcc._vhf = mlno._vhf
        if mlno.kwargs_imp is not None:
            mcc = mcc.set(**mlno.kwargs_imp)
        time0 = time.perf_counter()
        (eorb_mp2, eorb_cc), t1, t2 =\
            mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
        time1 = time.perf_counter()
        lnocc_time = time1 - time0

        print(f'LNO-MP2 Orbital Energy:  {eorb_mp2:.8f}')
        print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')
        print(f"LNO-CCSD time:           {lnocc_time:.2f} s")

    #     las_center[ifrag] = loc_ctr
    #     las_size[ifrag] = lno_tot
    #     lno_emp2[ifrag] = eorb_mp2
    #     lno_ecc[ifrag] = eorb_cc
    #     ccsd_time[ifrag] = lnocc_time

    #     # project onto center lo space
    #     # <lno_actocc|orbloc> <orbloc|lno_actocc>
    #     if spin_type == "unrestricted":
    #         prjlo = [uocc_loc[0] @ uocc_loc[0].T.conj(),
    #                  uocc_loc[1] @ uocc_loc[1].T.conj()]
    #         qmc_options["trial"] = trial_base
    #         if 'ad' not in trial_base:
    #             if lno_elec_type == 'alpha':
    #                 qmc_options["trial"] += '_alpha'
    #             elif lno_elec_type == 'beta':
    #                 qmc_options["trial"] += '_beta'
    #     else:
    #         prjlo = uocc_loc @ uocc_loc.T.conj()

    #     qmc_options["seed"] = seeds[ifrag]
    #     prep.prep_afqmc_integral(
    #         mf,
    #         lno_coeff,
    #         t1,
    #         t2,
    #         lno_frozen,
    #         prjlo,
    #         qmc_options,
    #         chol_cut=chol_cut
    #         )
        
    #     lno_afqmc.run_lnoafqmc(qmc_options)
    #     outfile = f'fragment.out{run_frag_list[ifrag]+1}'
    #     os.system(f'mv afqmc.out {outfile}')
    #     with open(outfile, "r") as f:
    #         for line in f:
    #             if "Blocked AFQMC/pt2CCSD Orbital Energy" in line:
    #                 eorb_afqmc = float(line.split()[-3])
    #                 eorb_afqmc_err = float(line.split()[-1])
    #             if "total run time" in line:
    #                 lnoqmc_time = float(line.split()[-1])
    #     lno_eqmc[ifrag] = eorb_afqmc
    #     lno_eqmc_err[ifrag] = eorb_afqmc_err
    #     qmc_time[ifrag] = lnoqmc_time

    #     header = f' Fragment{run_frag_list[ifrag]+1} Results '
    #     width = 80  # pick a consistent total width
    #     with open(outfile, 'a') as f:
    #         f.write('\n')
    #         f.write(f'{header:=^{width}}\n')
    #         f.write("\t LNO Center " + loc_ctr + "\n")
    #         f.write('-' * width + '\n')
    #         f.write(f'\t LNO-Active Space electrons: {nactocc} | orbitals: {nactocc+nactvir} \n')
    #         f.write(f'\t LNO-MP2 Orbital Energy:   {eorb_mp2:.8f} \n')
    #         f.write(f'\t LNO-CCSD Orbital Energy:  {eorb_cc:.8f} \n')
    #         f.write(f'\t LNO-AFQMC Orbital Energy: {eorb_afqmc:.6f} +/- {eorb_afqmc_err:.6f} \n')
    #         f.write(f'\t LNO-CCSD Time:  {lnocc_time:.2f} \n')
    #         f.write(f'\t LNO-AFQMC Time: {lnoqmc_time:.2f} \n')
    #         f.write('=' * width + '\n')
    #     jax.clear_caches()
    #     gc.collect()

    # las_size = np.array(las_size, dtype=np.int32)
    # las_max = las_size.max()
    # # convert to list of string for print
    # las_size = list(map(lambda row: f"{row}", las_size))
    # e_mp2 = np.sum(lno_emp2)
    # e_ccsd = np.sum(lno_ecc)
    # e_afqmc = np.sum(lno_eqmc)
    # e_afqmc_err = np.sqrt(np.sum(lno_eqmc_err**2))
    # tot_ccsd_time = np.sum(ccsd_time)
    # tot_qmc_time = np.sum(qmc_time)

    # with open(f'lno_result.out', 'w') as f:
    #     width = 100
    #     f.write('=' * width + '\n')
    #     f.write(f'{"LNO-AFQMC Results":^{width}}\n')
    #     f.write('=' * width + '\n')

    #     f.write(f'{"Frag":>4s}  {"LAS Center":>14s}  {"LAS_SIZE":>8s}  '
    #             f'{"E(MP2)":>10s}  {"E(CCSD)":>10s}  '
    #             f'{"E(AFQMC)":>10s}  {"Error":>8s}  '
    #             f'{"t(CCSD)":>8s}  {"t(AFQMC)":>8s}\n')
    #     f.write('-' * width + '\n')
        
    #     for n, i in enumerate(run_frag_list):
    #         f.write(f"{i+1:4d}  {las_center[n]:>14s}  {las_size[n]:8s}  "
    #                 f"{lno_emp2[n]:10.8f}  {lno_ecc[n]:10.8f}  "
    #                 f"{lno_eqmc[n]:10.6f}  {lno_eqmc_err[n]:8.6f}  "
    #                 f"{ccsd_time[n]:8.2f}  {qmc_time[n]:8.2f}\n")
        
    #     f.write('-' * width + '\n')
    #     f.write(f'{"Summarize Fragments":^{width}}\n')
    #     f.write('-' * width + '\n')
    #     lno_thresh_str = "[" + ", ".join(f"{x:.2e}" for x in lno_thresh) + "]"
    #     f.write(f'{"LNO-Thresh":<20} {"Max LAS":>8} '
    #             f'{"E[MP2]":>12} {"E[CCSD]":>12} '
    #             f'{"E[AFQMC]":>10} {"Err[AFQMC]":>10} '
    #             f'{"CCSD-Time":>10} {"AFQMC-Time":>10}\n')

    #     f.write(f'{lno_thresh_str:<20} {las_max:>8} '
    #             f'{e_mp2:>12.8f} {e_ccsd:>12.8f} '
    #             f'{e_afqmc:>10.6f} {e_afqmc_err:>10.6f} '
    #             f'{tot_ccsd_time:>10.2f} {tot_qmc_time:>10.2f}\n')
        
    #     f.write('=' * width + '\n\n')

    return None

In [25]:
options = {
           'eql_time': 10,
           'n_blocks': 50,
           'n_walkers': 300,
           'mix_precision': True,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

In [28]:
run_afqmc(mf,
          lo_coeff = iao_coeff, 
          frag_lolist = iao_frag_list,
          nfrozen = mycc.frozen,
          thresh = 1e-5,
          qmc_options = options, 
          chol_cut = 1e-5, 
          target_sto_error = 1e-3, 
          run_frag_list = [0], 
          atom_group = atm_center,
          plot_las = False
          )



====================== unrestricted LNO-FRAGMENT 1/(1,6) =======================
Center Atom O
PySCF NumPy Threads = 16
[[{'thresh': 0.0001, 'pct_occ': None, 'norb': None}, {'thresh': 1e-05, 'pct_occ': None, 'norb': None}], [{'thresh': 0.0001, 'pct_occ': None, 'norb': None}, {'thresh': 1e-05, 'pct_occ': None, 'norb': None}]]
LNO-Frgament Spin Type = mixed
LAS occupied orbitals:  [7, 7]
LAS virtual orbitals:   [27, 27]
LAS total size:         [34, 34]
LNO-MP2 Orbital Energy:  -0.17002261
LNO-CCSD Orbital Energy: -0.17750979
LNO-CCSD time:           0.42 s


In [30]:
options = {
           'eql_time': 10,
           'n_blocks': 50,
           'n_walkers': 300,
           'mix_precision': True,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

mf = mf
lo_coeff = iao_coeff 
frag_lolist = iao_frag_list
nfrozen = mycc.frozen
thresh = 1e-5
qmc_options = options
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = [0]
atom_group = None
plot_las = False

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)

mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)
print(f"Run {nfrag_run} fragments out of {nfrag_tot} total fragments")

lno_pct_occ = [None, None]
lno_norb = [[None, None]] * nfrag_run

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_run,), 
                        minval=0, 
                        maxval=100*nfrag_run
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
# las_size = np.zeros(nfrag, dtype='int32')
lno_emp2 = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol
print(lno_thresh)
print(lno_pct_occ)
print(lno_norb)

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[run_frag_list[ifrag]]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')

        if loc_ctr is None:
            ao_max_a = prep.ao_comp(mf, orbloc[0])
            ao_max_b = prep.ao_comp(mf, orbloc[1])
            loc_ctr = ao_max_a + ao_max_b
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = [nactocc_a, nactocc_b]
        nactvir = [nactvir_a, nactvir_b]
        lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
        lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
        lno_active = [lno_active_a, lno_active_b]
        lno_tot = [len(lno_active_a), len(lno_active_b)]
        # print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        print(f'LNO-Frgament Spin Type = restricted')
        if loc_ctr is None:
            loc_ctr = prep.ao_comp(mf, orbloc)
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        lno_tot = len(lno_active)
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx = run_frag_list[ifrag]+1)

    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')
    print(f"LNO-CCSD time:           {lnocc_time:.2f} s")

    las_center[ifrag] = loc_ctr
    las_size[ifrag] = lno_tot
    lno_emp2[ifrag] = eorb_mp2
    lno_ecc[ifrag] = eorb_cc
    ccsd_time[ifrag] = lnocc_time

    # project onto center lo space
    # <lno_actocc|orbloc> <orbloc|lno_actocc>
    if spin_type == "unrestricted":
        prjlo = [uocc_loc[0] @ uocc_loc[0].T.conj(),
                    uocc_loc[1] @ uocc_loc[1].T.conj()]
        qmc_options["trial"] = trial_base
        if 'ad' not in trial_base:
            if lno_elec_type == 'alpha':
                qmc_options["trial"] += '_alpha'
            elif lno_elec_type == 'beta':
                qmc_options["trial"] += '_beta'
    else:
        prjlo = uocc_loc @ uocc_loc.T.conj()

    qmc_options["seed"] = seeds[ifrag]
    prep.prep_afqmc_integral(
        mf,
        lno_coeff,
        t1,
        t2,
        lno_frozen,
        prjlo,
        qmc_options,
        chol_cut=chol_cut
        )
    
    lno_afqmc.run_lnoafqmc(qmc_options)

Run 1 fragments out of 6 total fragments
[0.0001, 1e-05]
[None, None]
[[None, None]]


====================== unrestricted LNO-FRAGMENT 1/(1,6) =======================
PySCF NumPy Threads = 16
LNO-Frgament Spin Type = mixed
LNO Center 0 O 1s    0 O 1s    
LAS occupied orbitals:  [7, 7]
LAS virtual orbitals:   [27, 27]
LAS total size:         [34, 34]
LNO-MP2 Orbital Energy:  -0.17002261
LNO-CCSD Orbital Energy: -0.17750979
LNO-CCSD time:           0.40 s
Calculating Effective Active Space One-electron Integrals
Building JK matrix
build JK time: 0.267686 s
build ecore time: 0.104699 s
build h1eff time: 0.422986 s
Generating Cholesky Integrals
Constracting cLAS that span both Alpha and Beta active space
Naive cLAS Shape:  (48, 68)
Orthonormalize cLAS shape: (48, 48)
Smallest cLAS SVD Singular values: 2.218826569978176e-09
cLAS projection torr: 1e-08
Minimum size of cLAS to span both Alpha and Beta LAS: 45
cLAS projection loss: (3.72e-09, 3.72e-09)
True Common LAS Shape:  (48, 45)
Composi

In [31]:
from functools import partial
print = partial(print, flush=True)

print("\nAFQMC Started")

from afqmc import config
config.setup_jax()

import time
import numpy as np
from jax import random
from jax import numpy as jnp

from afqmc import sampling
from afqmc.lno_afqmc import prep

init_time = time.time()

ham_data, prop, trial, wave_data, sampler, options = (prep.prep_afqmc_run())

print(f"Trial: {trial}")
print(f"Sampler: {sampler}")

### initialize propagation
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
    
ham_data = trial._build_measurement_intermediates(ham_data, wave_data)
ham_data = prop._build_propagation_intermediates(ham_data, trial, wave_data)
prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers = None)

if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )

prop_data["key"] = random.PRNGKey(options["seed"])
prop_data["n_killed_walkers"] = 0
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]
e_init = prop_data["e_estimate"]

e0, t1olp, eorb, t2eorb, t2orb, e0bar \
    = trial.calc_eorb_pt2(prop_data['walkers'], ham_data, wave_data)

e0 = jnp.real(e0)[0]
t1olp = jnp.real(t1olp)[0]
eorb = jnp.real(eorb)[0]
t2eorb = jnp.real(t2eorb)[0]
t2orb = jnp.real(t2orb)[0]
e0bar = jnp.real(e0bar)[0]
eorb_pt = jnp.real(eorb/t1olp + t2eorb/t1olp - t2orb*e0bar/t1olp**2)

print("\nEquilibration")
print(f"Initial Orbital energy: {eorb_pt:.6f}")
print(f"{'inv_T':>5s}  {'energy':>12s}  {'runTime':>8s}")
print(f"{0.:5.2f}  {e0:12.6f}  {time.time() - init_time:8.2f}")

sampler_eq = sampling.sampler(
    n_prop_steps=50,
    n_chol = sampler.n_chol
    )

block_time = prop.dt * sampler_eq.n_prop_steps
neql_block = int(-(-options["eql_time"] // block_time))

for n in range(1, neql_block+1):
    prop_data, (wt, e) \
        = sampler_eq.block_sample(prop_data, ham_data, prop, trial, wave_data)

    if (n+1) % (min(max(neql_block // 10, 1), 20)) == 0 and n > 0:
        nkill = prop_data["n_killed_walkers"]
        print(f"{(n+1)*block_time:5.2f}  {e:12.6f}  {time.time() - init_time:8.2f}")


AFQMC Started
Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      6.17.0-35-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu

QMC System
Number of electrons:        (7, 7)
Number of orbitals:         (34, 34)
Number of Cholesky Vectors: 218
Maximum memory per walker:            6.67 MB
Maximum number of Cholesky per chunk: 188
Number of Cholesky chunks:            2
Number of Cholesky per chunk:         109
Number of padding Cholesky:           0

QMC Parameters
eql_time        -              10
n_blocks        -              50
n_walkers       -             300
mix_precision   -            True
seed            -              89
walker_type     -             uhf
trial           -        upt2ccsd
max_error       -       0.0004082
dt              -           0.005
n_prop_steps    -              50
n_batch         -               1
max_mem

In [35]:
def decompose_ut2(t2, thresh=1e-8):
    t2aa, t2ab, t2bb = t2
    nocca, nvira, noccb, nvirb = t2ab.shape

    npaira = nocca * nvira
    npairb = noccb * nvirb

    assert t2aa.shape == (nocca, nvira, nocca, nvira)
    assert t2bb.shape == (noccb, nvirb, noccb, nvirb)

    t2aa = t2aa.reshape(npaira, npaira)
    t2ab = t2ab.reshape(npaira, npairb)
    t2bb = t2bb.reshape(npairb, npairb)

    # Symmetric full t2 
    # [[ t2aa/2  t2ab   ]]
    # [[ t2ab^T  t2bb/2 ]]
    # t2full = np.zeros((npaira + npairb, npaira + npairb))
    # t2full[:npaira, :npaira] = 0.5 * t2aa
    # t2full[npaira:, :npaira] = t2ab.T
    # t2full[:npaira, npaira:] = t2ab
    # t2full[npaira:, npaira:] = 0.5 * t2bb
    # t2full = jnp.array(t2full)
    t2full = jnp.block([[0.5 * t2aa, t2ab],
                        [t2ab.T, 0.5 * t2bb]])
    # t2 = LL^T
    e_val, e_vec = jnp.linalg.eigh(t2full)

    # Keep only important modes
    mask = jnp.abs(e_val) > thresh
    e_val_trunc = e_val[mask]
    e_vec_trunc = e_vec[:, mask]
    
    tau = e_vec_trunc @ jnp.diag(jnp.sqrt(e_val_trunc + 0.0j))
    err = jnp.linalg.norm(t2full - tau @ tau.T)
    assert err < 10 * thresh, f"err={err}, threshold={thresh}"

    # alpha/beta operators for HS
    # Summation on the left to have a list of operators
    taua = tau.T[:,:npaira]
    taub = tau.T[:, npaira:]
    taua = taua.reshape(-1, nocca, nvira)
    taub = taub.reshape(-1, noccb, nvirb)

    return (taua, taub)

In [42]:
def decompose_t2(trial, t2, thresh=1e-8):
    # adapted from Yann
    norba, norbb = trial.norb
    nocca, noccb = trial.nelec
    nvira, nvirb = (norba - nocca, norbb - noccb)

    # Number of excitation pairs
    nex_a = nocca * nvira
    nex_b = noccb * nvirb

    t2aa, t2ab, t2bb = t2

    assert t2aa.shape == (nocca, nvira, nocca, nvira)
    assert t2ab.shape == (nocca, nvira, noccb, nvirb)
    assert t2bb.shape == (noccb, nvirb, noccb, nvirb)

    print('Decomposing Unrestricted T2 amplitudes')

    t2aa = t2aa.reshape(nex_a, nex_a)
    t2ab = t2ab.reshape(nex_a, nex_b)
    t2bb = t2bb.reshape(nex_b, nex_b)

    # Symmetric full t2 
    # [[ t2aa/2  t2ab   ]]
    # [[ t2ab^T  t2bb/2 ]]
    t2full = np.zeros((nex_a + nex_b, nex_a + nex_b))
    t2full[:nex_a, :nex_a] = 0.5 * t2aa
    t2full[nex_a:, :nex_a] = t2ab.T
    t2full[:nex_a, nex_a:] = t2ab
    t2full[nex_a:, nex_a:] = 0.5 * t2bb
    t2full = jnp.array(t2full)

    # t2 = LL^T
    e_val, e_vec = jnp.linalg.eigh(t2full)

    # Keep only important modes
    mask = jnp.abs(e_val) > thresh
    e_val_trunc = e_val[mask]
    e_vec_trunc = e_vec[:, mask]
    
    tau = e_vec_trunc @ jnp.diag(np.sqrt(e_val_trunc + 0.0j))
    err = jnp.linalg.norm(t2full - tau @ tau.T)
    assert err < 10 * thresh, f"err={err}, threshold={thresh}"
    print(f'Throw {len(e_val)-len(e_val_trunc)} vectors in T2 deomposition')
    print(f'SVD cutoff = {thresh:.2e} | error = {err:.2e}')
    print(f'number of T2 decomposition vectors {len(e_val_trunc)}')

    # alpha/beta operators for HS
    # Summation on the left to have a list of operators
    taua = tau.T[:,:nex_a]
    taub = tau.T[:, nex_a:]
    taua = taua.reshape(-1, nocca, nvira)
    taub = taub.reshape(-1, noccb, nvirb)

    return [taua, taub]

In [47]:
amplitudes = np.load("amplitudes.npz")
# t1a = jnp.array(amplitudes["t1a"])
# t1b = jnp.array(amplitudes["t1b"])
t2aa = jnp.array(amplitudes["t2aa"])
t2ab = jnp.array(amplitudes["t2ab"])
t2bb = jnp.array(amplitudes["t2bb"])
print(t2aa.shape)
print(t2ab.shape)
print(t2bb.shape)

(7, 27, 7, 27)
(7, 27, 7, 27)
(7, 27, 7, 27)


In [49]:
[taua, taub] = decompose_t2(trial, t2=(t2aa,t2ab,t2bb), thresh=1e-7)

Decomposing Unrestricted T2 amplitudes
Throw 2 vectors in T2 deomposition
SVD cutoff = 1.00e-07 | error = 3.19e-08
number of T2 decomposition vectors 376


In [50]:
taua.shape

(376, 7, 27)

In [53]:
import opt_einsum as oe
t2aa_rec = oe.contract("gia,gjb->iajb",taua,taua,backend="jax") * 2
print((t2aa-t2aa_rec).max())

(1.1296379376319688e-08+0j)


In [57]:
from afqmc import t2_tools

taua = t2_tools.decompose_rt2(t2=t2aa, thresh=1e-7)
t2aa_rec = oe.contract("gia,gjb->iajb",taua,taua,backend="jax")
print((t2aa-t2aa_rec).max())

(2.963368412291158e-08+0j)


In [58]:
taua.shape

(186, 7, 27)

In [ ]:
# @partial(jit, static_argnums=0)
import opt_einsum as oe
from jax import lax

def _t2eorb_tc(self, walker_up, walker_dn, ham_data, wave_data):
    """use chunked cholesky for two-body terms"""
    if self.mix_precision:
        rtype = jnp.float32
        ctype = jnp.complex64
    else:
        rtype = jnp.float64
        ctype = jnp.complex128
    
    nchol_chunk = self.nchol_chunk
    norb_a, norb_b = self.norb
    nocc_a, nocc_b = self.nelec
    h1a, h1b = ham_data["h1bar"]
    t2aa, t2ab = wave_data["t2aa"], wave_data["t2ab"]
    t2ba, t2bb = wave_data["t2ba"], wave_data["t2bb"]
    chol_a, chol_b = ham_data["chol_bar"]
    chol_a = chol_a.reshape(-1, norb_a, norb_a)
    chol_b = chol_b.reshape(-1, norb_b, norb_b)
    rot_chol_a = chol_a[:, :nocc_a, :]
    rot_chol_b = chol_b[:, :nocc_b, :]

    green_a = (walker_up.dot(jnp.linalg.inv(walker_up[:nocc_a, :]))).T  # G_ip
    green_b = (walker_dn.dot(jnp.linalg.inv(walker_dn[:nocc_b, :]))).T
    green_occ_a = green_a[:, nocc_a:]
    green_occ_b = green_b[:, nocc_b:]
    greenp_a = jnp.vstack((green_occ_a, -jnp.eye(norb_a - nocc_a)))
    greenp_b = jnp.vstack((green_occ_b, -jnp.eye(norb_b - nocc_b)))

    # 1 body energy
    hg_a = oe.contract("pj,pj->", h1a[:nocc_a, :], green_a, backend="jax")
    hg_b = oe.contract("pj,pj->", h1b[:nocc_b, :], green_b, backend="jax")
    e1_0 = hg_a + hg_b  # <HF|h1|walker>/<HF|walker>

    # double excitations
    # i <-> j does not have anti-sym in LNO!!!
    t2g_aa_a_c = oe.contract("iajb,ia->jb", t2aa, green_occ_a, backend="jax") / 4
    t2g_aa_a_e = oe.contract("iajb,ja->ib", t2aa, green_occ_a, backend="jax") / 4
    t2g_bb_b_c = oe.contract("iajb,ia->jb", t2bb, green_occ_b, backend="jax") / 4
    t2g_bb_b_e = oe.contract("iajb,ja->ib", t2bb, green_occ_b, backend="jax") / 4
    t2g_ab_a = oe.contract("iajb,ia->jb", t2ab, green_occ_a, backend="jax") / 2
    t2g_ab_b = oe.contract("iajb,jb->ia", t2ab, green_occ_b, backend="jax") / 2
    t2g_ba_a = oe.contract("iajb,jb->ia", t2ba, green_occ_a, backend="jax") / 2
    t2g_ba_b = oe.contract("iajb,ia->jb", t2ba, green_occ_b, backend="jax") / 2
    gt2g_aa = oe.contract("jb,jb->", t2g_aa_a_c, green_occ_a, backend="jax")
    gt2g_bb = oe.contract("jb,jb->", t2g_bb_b_c, green_occ_b, backend="jax")
    gt2g_ab = oe.contract("jb,jb->", t2g_ab_a, green_occ_b, backend="jax")
    gt2g_ba = oe.contract("jb,jb->", t2g_ba_b, green_occ_a, backend="jax")
    gt2g = (gt2g_aa + gt2g_bb) * 2 + (gt2g_ab + gt2g_ba)
    e1_2_1 = gt2g * e1_0

    # t_iajb G_ia G_jq Gp_pb
    t2_green_aaa_c = oe.contract('pb,jb,jq->pq', greenp_a, t2g_aa_a_c, green_a, backend="jax")
    t2_green_aaa_e = oe.contract('pb,ib,iq->pq', greenp_a, t2g_aa_a_e, green_a, backend="jax")
    t2_green_bbb_c = oe.contract('pb,jb,jq->pq', greenp_b, t2g_bb_b_c, green_b, backend="jax")
    t2_green_bbb_e = oe.contract('pb,ib,iq->pq', greenp_b, t2g_bb_b_e, green_b, backend="jax")
    t2_green_aba = oe.contract('pa,ia,iq->pq', greenp_a, t2g_ab_b, green_a, backend="jax")
    t2_green_baa = oe.contract('pb,jb,jq->pq', greenp_a, t2g_ba_b, green_a, backend="jax")
    t2_green_bab = oe.contract('pa,ia,iq->pq', greenp_b, t2g_ba_a, green_b, backend="jax")
    t2_green_abb = oe.contract('pb,jb,jq->pq', greenp_b, t2g_ab_a, green_b, backend="jax")
    t2_green_aaa = 2 * (t2_green_aaa_c - t2_green_aaa_e)
    t2_green_bbb = 2 * (t2_green_bbb_c - t2_green_bbb_e)
    e1_2_2_a = -oe.contract("pq,pq->", t2_green_aaa + t2_green_aba + t2_green_baa, h1a, backend="jax")
    e1_2_2_b = -oe.contract("pq,pq->", t2_green_bbb + t2_green_bab + t2_green_abb, h1b, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <HF|T2 h1|walker>/<HF|walker>

    # two body energy — chunked over Cholesky auxiliary index
    nchol = rot_chol_a.shape[0]
    nchol_chunk = self.nchol_chunk
    nchunks = -(-nchol // nchol_chunk)
    npad = nchunks * nchol_chunk - nchol

    chol_a = jnp.concatenate([chol_a, jnp.zeros((npad, norb_a, norb_a))], axis=0)
    chol_b = jnp.concatenate([chol_b, jnp.zeros((npad, norb_b, norb_b))], axis=0)
    rot_chol_a = jnp.concatenate([rot_chol_a, jnp.zeros((npad, nocc_a, norb_a))], axis=0)
    rot_chol_b = jnp.concatenate([rot_chol_b, jnp.zeros((npad, nocc_b, norb_b))], axis=0)

    chol_a = chol_a.reshape(nchunks, nchol_chunk, norb_a, norb_a)
    chol_b = chol_b.reshape(nchunks, nchol_chunk, norb_b, norb_b)
    rot_chol_a = rot_chol_a.reshape(nchunks, nchol_chunk, nocc_a, norb_a)
    rot_chol_b = rot_chol_b.reshape(nchunks, nchol_chunk, nocc_b, norb_b)

    # combined intermediates so we don't recompute them each chunk
    t2_green_a_tot = 2 * t2_green_aaa + 2 * (t2_green_aba + t2_green_baa)
    t2_green_b_tot = 2 * t2_green_bbb + 2 * (t2_green_bab + t2_green_abb)

    def scan_chunk(carry, x):
        chol_a_c, rot_chol_a_c, chol_b_c, rot_chol_b_c = x

        gl_a = oe.contract("ir,gpr->gip", green_a, chol_a_c, backend="jax")
        gl_b = oe.contract("ir,gpr->gip", green_b, chol_b_c, backend="jax")
        tr_gl_a = oe.contract("gii->g", gl_a[:, :nocc_a, :nocc_a], backend="jax")
        tr_gl_b = oe.contract("gii->g", gl_b[:, :nocc_b, :nocc_b], backend="jax")
        gl_c = tr_gl_a + tr_gl_b
        e2_0_c = oe.contract('g,g->', gl_c, gl_c) / 2.0
        e2_0_e = -(oe.contract("gij,gji->", gl_a[:, :nocc_a, :nocc_a], gl_a[:, :nocc_a, :nocc_a], backend="jax")
                + oe.contract("gij,gji->", gl_b[:, :nocc_b, :nocc_b], gl_b[:, :nocc_b, :nocc_b], backend="jax")) / 2.0
        carry[0] += e2_0_c + e2_0_e

        # double excitations
        lt2g_a = oe.contract("gpq,pq->g", chol_a_c, t2_green_a_tot, backend="jax")
        lt2g_b = oe.contract("gpq,pq->g", chol_b_c, t2_green_b_tot, backend="jax")
        carry[1] += -oe.contract('g,g->', lt2g_a + lt2g_b, gl_c, backend="jax") / 2.0

        lt2_green_a = oe.contract("gpi,ji->gpj", rot_chol_a_c, t2_green_a_tot, backend="jax")
        lt2_green_b = oe.contract("gpi,ji->gpj", rot_chol_b_c, t2_green_b_tot, backend="jax")
        carry[2] += (
            (oe.contract("gip,gip->", gl_a.astype(ctype), lt2_green_a.astype(ctype), backend="jax")
            + oe.contract("gip,gip->", gl_b.astype(ctype), lt2_green_b.astype(ctype), backend="jax")) / 2
            ).astype(jnp.complex128)

        glgp_a = oe.contract("gip,pa->gia", gl_a, greenp_a, backend="jax")
        glgp_b = oe.contract("gip,pa->gia", gl_b, greenp_b, backend="jax")

        l2t2_aa_a = oe.contract("gia,iajb->gjb", glgp_a.astype(ctype), t2aa.astype(rtype), backend="jax")
        l2t2_ab_a = oe.contract("gia,iajb->gjb", glgp_a.astype(ctype), t2ab.astype(rtype), backend="jax")
        l2t2_ba_b = oe.contract("gia,iajb->gjb", glgp_b.astype(ctype), t2ba.astype(rtype), backend="jax")
        l2t2_bb_b = oe.contract("gia,iajb->gjb", glgp_b.astype(ctype), t2bb.astype(rtype), backend="jax")
        
        l2t2_aa = 0.5 * oe.contract("gjb,gjb->", 
                                    l2t2_aa_a.astype(ctype), 
                                    glgp_a.astype(ctype), 
                                    backend="jax").astype(jnp.complex128)
        l2t2_ab = 0.5 * oe.contract("gjb,gjb->", 
                                    l2t2_ab_a.astype(ctype), 
                                    glgp_b.astype(ctype), 
                                    backend="jax").astype(jnp.complex128)
        l2t2_ba = 0.5 * oe.contract("gjb,gjb->", 
                                    l2t2_ba_b.astype(ctype), 
                                    glgp_a.astype(ctype), 
                                    backend="jax").astype(jnp.complex128)
        l2t2_bb = 0.5 * oe.contract("gjb,gjb->", 
                                    l2t2_bb_b.astype(ctype), 
                                    glgp_b.astype(ctype), 
                                    backend="jax").astype(jnp.complex128)
        carry[3] += (l2t2_aa + l2t2_ab + l2t2_ba + l2t2_bb).astype(jnp.complex128)

        return carry, 0.0

    [e2_0, e2_2_2_1, e2_2_2_2, e2_2_3], _ \
        = lax.scan(scan_chunk, [0.0, 0.0, 0.0, 0.0], (chol_a, rot_chol_a, chol_b, rot_chol_b))

    e2_2_1 = e2_0 * gt2g
    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3  # <HF|T2 h2|walker>/<HF|walker>

    t2orb = gt2g  # <HF|T1+T2|walker>/<HF|walker>
    e12bar = e1_0 + e2_0  # <HF|h1+h2|walker>/<HF|walker>
    t2eorb = e1_2 + e2_2  # <HF|T2(h1+h2)|walker>/<HF|walker>

    return t2eorb, t2orb, e12bar